In [ ]:
import pandas as pd
import sys

sys.path.append("../src")
from data_loader import load_data
from data_processing import (
    RFMTransformer, HighRiskLabeler, merge_target
)



In [ ]:
df = load_data(
    "../data/data.csv"
)
print("Shape:", df.shape)

df.head()

2026-06-01 23:08:46,812 - INFO - Dataset loaded successfully


Shape: (95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [ ]:
rfm = (
    RFMTransformer()
    .fit_transform(df)
)

rfm.head()

,CustomerId,Recency,Frequency,Monetary
0,CustomerId_1,84,1,-10000.0
1,CustomerId_10,84,1,-10000.0
2,CustomerId_1001,90,5,20000.0
3,CustomerId_1002,26,11,4225.0
4,CustomerId_1003,12,6,20000.0


This RFM table is crucial. For each customer, we've summarized their engagement:
-   **Recency:** How recently they've transacted. High recency (larger number) indicates less recent activity, potentially higher risk.
-   **Frequency:** How often they transact. Low frequency is a key indicator of disengagement.
-   **Monetary:** Their total spending. Low monetary value suggests less valuable customers.
This aggregated view is the basis for segmenting your customer base and pinpointing those at risk.

In [ ]:
labeler = HighRiskLabeler()

rfm_clustered = (
    labeler.fit_transform(rfm)
)

rfm_clustered.head()

,CustomerId,Recency,Frequency,Monetary,Cluster,is_high_risk
0,CustomerId_1,84,1,-10000.0,0,1
1,CustomerId_10,84,1,-10000.0,0,1
2,CustomerId_1001,90,5,20000.0,0,1
3,CustomerId_1002,26,11,4225.0,1,0
4,CustomerId_1003,12,6,20000.0,1,0


Here, we've gone a step further. We've clustered your customers into distinct groups (`Cluster` column) based on their RFM behavior. More importantly, we've identified the 'high-risk' cluster and assigned a `1` to the `is_high_risk` column for these customers. This direct labeling is the critical step in preparing your data for a predictive model that can learn to spot these at-risk customers.

In [ ]:
rfm_clustered[
    "is_high_risk"
].value_counts()

is_high_risk
0    2316
1    1426
Name: count, dtype: int64

This count gives you an immediate overview of your customer risk landscape. Out of your unique customers, `1,426` are currently classified as 'high-risk' based on their RFM profile. This number is vital for sizing your target audience for retention campaigns or further investigation.

In [ ]:
final_df = merge_target(
    df,
    rfm_clustered
)

final_df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult,is_high_risk
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0,1
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0,0


By merging the `is_high_risk` label back into the original transaction data, your `final_df` is now your go-to dataset for predictive modeling. Every single transaction now implicitly tells you whether the customer involved is considered high-risk. This is the dataset you will use to train a model that learns the patterns associated with high-risk behavior.

In [ ]:
final_df[
    "is_high_risk"
].value_counts()

is_high_risk
0    84653
1    11009
Name: count, dtype: int64

This distribution shows that `11,009` of your transactions are linked to customers we've identified as high-risk. This provides a transaction-level view of risk, which can be useful for understanding the volume of transactions influenced by at-risk customers, beyond just the count of unique customers.

In [ ]:
final_df.to_csv(
    "../data/processed/credit_risk_with_target.csv",
    index=False
)

print(
    "Dataset saved successfully."
)

Dataset saved successfully.


Your enriched dataset, `credit_risk_with_target.csv`, is now safely saved! This means all the hard work of identifying high-risk customers is preserved, and you can easily load this file into any analytics or machine learning platform to immediately begin building your predictive models or conducting deeper analysis without re-running previous steps.

### Insight for the User

This notebook has successfully taken raw transaction data and transformed it into a valuable asset for customer relationship management and risk assessment. Here's what has been achieved and its utility:

1.  **RFM Metrics Calculation**: By computing Recency, Frequency, and Monetary values, we've gained a comprehensive understanding of each customer's engagement and value to the business. This summarization is crucial for customer segmentation.

2.  **Customer Segmentation**: Using K-Means clustering on the RFM profiles, we've grouped customers into distinct segments. This allows for tailored strategies for different customer behaviors.

3.  **High-Risk Customer Identification**: A critical outcome is the identification of a 'high-risk' customer segment. These are typically customers with low engagement (low frequency, low monetary value, high recency).

4.  **Actionable Target Variable**: The new `is_high_risk` column serves as a direct target variable. This means you can now:
    *   **Train Predictive Models**: Develop machine learning models to predict which new or existing customers are likely to fall into the high-risk category.
    *   **Proactive Interventions**: Implement targeted marketing campaigns, special offers, or customer service outreach to engage these high-risk customers before they churn or reduce their spending.
    *   **Resource Allocation**: Optimize resource allocation by focusing retention efforts on the most vulnerable customer segments.

5.  **Persisted Data**: The final processed dataset, `credit_risk_with_target.csv`, is saved and ready for immediate use in any machine learning framework or business intelligence tool. This ensures reproducibility and ease of further analysis.